# 06 · Evaluate fine-tuned models B, C, D (SGLang + AuditKIT) and benchmark serving of D
**Purpose:** serve each merged model with SGLang (one at a time), run the same 500 TEST items with the same prompt/parser/decoding as model A, then `sglang.bench_serving` on D.
**Inputs:** `artifacts/models/{B,C,D}_merged/`. **Outputs:** `results/eval_{B,C,D}.json`, `results/predictions_{B,C,D}.jsonl`, `results/serving_D.json`.
**Runtime (T4):** ~5 min per model + ~5 min benchmark.

In [ ]:
import os, sys, json, subprocess
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
C.ensure_env()
cfg = C.load_config(); FORCE = False
GPU, P = C.gpu_profile(cfg)
MODELS = C.ART / "models"
print("profile", GPU, "| merged models:", sorted(p.name for p in MODELS.glob("*_merged")))

In [ ]:
for arm in ("B", "C", "D"):
    path = MODELS / f"{arm}_merged"
    if not path.exists():
        print(f"[{arm}] missing {path} — run notebook 05"); continue
    if FORCE or not (C.RESULTS / f"eval_{arm}.json").exists():
        C.start_server(str(path), arm, P["sglang_args"], mem_fraction=P["mem_fraction"])
        C.evaluate_on_test(arm, arm, cfg)
        C.stop_server()

In [ ]:
# Serving performance of D: sglang.bench_serving, random prompts sized like PubMedQA (≈512 in / 128 out), 32 concurrent.
out = C.RESULTS / "serving_D.jsonl"
if FORCE or not out.exists():
    base = C.start_server(str(MODELS / "D_merged"), "D", P["sglang_args"], mem_fraction=P["mem_fraction"])
    cmd = [C.SGLANG_PY, "-m", "sglang.bench_serving", "--backend", "sglang", "--port", str(cfg["sglang"]["port"]),
           "--dataset-name", "random", "--random-input-len", "512", "--random-output-len", "128", "--random-range-ratio", "1.0",
           "--num-prompts", "200", "--max-concurrency", "32", "--seed", str(cfg["seed"]), "--output-file", str(out)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-1500:])
    C.stop_server()
bench = C.read_jsonl(out)[-1]
keys = ["request_throughput", "input_throughput", "output_throughput", "median_e2e_latency_ms", "p99_e2e_latency_ms",
        "median_ttft_ms", "median_itl_ms", "completed", "max_concurrency"]
gpu_name = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
serving = {"gpu": gpu_name, **{k: bench.get(k) for k in keys}}
json.dump(serving, open(C.RESULTS / "serving_D.json", "w"), indent=1)
print(json.dumps(serving, indent=1))

In [ ]:
for m in ("A", "B", "C", "D", "T"):
    f = C.RESULTS / f"eval_{m}.json"
    if f.exists():
        r = json.load(open(f))
        print(f"{m}: acc={r['accuracy']:.3f} {[round(x, 3) for x in r['accuracy_ci95']]} macroF1={r['macro_f1']:.3f} "
              f"{[round(x, 3) for x in r['macro_f1_ci95']]} fmt_fail={r['format_failure_rate']:.3f}")
print("Saved:", sorted(p.name for p in C.RESULTS.glob("*.json")))